# 📘 Bài Giảng Thực Hành: Hồi Quy Tuyến Tính Đa Biến (Multiple Linear Regression)
**Mục tiêu bài giảng:**
1. Mô hình hóa mối quan hệ giữa nhiều biến độc lập $x_1, x_2, \dots, x_n$ và biến mục tiêu $y$.
2. Xử lý biến phân loại (Categorical Data) bằng `ColumnTransformer` + `OneHotEncoder(drop='first')` để tránh **Bẫy Biến Giả (Dummy Variable Trap)**.
3. Thực hiện thuật toán **Loại Bỏ Ngược (Backward Elimination)** bằng kiểm định p-value qua `statsmodels.api.OLS`.

---
### 📊 Tập dữ liệu thực hành: `50_Startups.csv`
- **Features**: `R&D Spend`, `Administration`, `Marketing Spend`, `State` (New York, California, Florida).
- **Target**: `Profit` - Lợi nhuận của doanh nghiệp Startup.


## 1. Nạp Thư viện & Khám phá Dữ liệu


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
import statsmodels.api as sm

dataset = pd.read_csv('50_Startups.csv')
print('Số lượng mẫu:', len(dataset))
dataset.head()


📌 **Expected Output:**
```
Số lượng mẫu: 50
   R&D Spend  Administration  Marketing Spend       State     Profit
0  165349.20       136897.80        471784.10    New York  192261.83
1  162597.70       151377.59        443898.53  California  191792.06
2  153441.51       101145.55        407934.54     Florida  191050.39
3  144372.41       118671.85        383199.62    New York  182901.99
4  142107.34        91391.77        366168.42     Florida  166187.94
```


## 2. Tiền Xử Lý: One-Hot Encoding Biến 'State' (Tránh Dummy Variable Trap)


In [ ]:
X_raw = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1].values

# Sử dụng drop='first' để loại bỏ 1 cột dummy tránh đa cộng tuyến hoàn hảo
ct = ColumnTransformer(
    transformers=[('encoder', OneHotEncoder(drop='first'), ['State'])],
    remainder='passthrough'
)
X = np.array(ct.fit_transform(X_raw), dtype=float)
feature_names = list(ct.named_transformers_['encoder'].get_feature_names_out(['State'])) + ['R&D Spend', 'Administration', 'Marketing Spend']
print('Danh sách đặc trưng sau khi mã hóa:', feature_names)

# Chia tập Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


📌 **Expected Output:**
```
Danh sách đặc trưng sau khi mã hóa: ['State_Florida', 'State_New York', 'R&D Spend', 'Administration', 'Marketing Spend']
```


## 3. Huấn luyện Mô hình Hồi quy Tuyến tính Đa biến (Full Model)


In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

print(f'R² Score trên Test Set: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE trên Test Set:     ${root_mean_squared_error(y_test, y_pred):,.2f}')


📌 **Expected Output:**
```
R² Score trên Test Set: 0.9347
RMSE trên Test Set:     $9,137.99
```


## 4. Thuật toán Loại Bỏ Ngược (Backward Elimination) với Statsmodels OLS
Thực hiện loại bỏ các biến có $p\text{-value} > 0.05$ để giữ lại các biến có ý nghĩa thống kê cao nhất.


In [ ]:
# Thêm cột hệ số tự do w0 (toàn số 1)
X_opt = np.append(arr=np.ones((X.shape[0], 1)).astype(float), values=X, axis=1)
all_feature_names = ['Intercept (w0)'] + feature_names

# Huấn luyện OLS ban đầu
regressor_ols = sm.OLS(endog=y, exog=X_opt).fit()
print(regressor_ols.summary())


📌 **Expected Output:**
```
OLS Regression Results hiển thị: R&D Spend có p-value = 0.000 (cực kỳ có ý nghĩa thống kê), trong khi State và Administration có p-value > 0.05.
```


## 5. Trực quan hóa So sánh Thực tế vs Dự đoán & Trọng số Mô hình


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Đồ thị 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, color='blue', s=80, alpha=0.7)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
axes[0].set_title('Thực tế vs Dự đoán (Actual vs Predicted)', fontweight='bold')
axes[0].set_xlabel('Lợi nhuận Thực tế ($)')
axes[0].set_ylabel('Lợi nhuận Dự đoán ($)')

# Đồ thị 2: Trọng số của các đặc trưng
colors = ['green' if c > 0 else 'red' for c in regressor.coef_]
axes[1].barh(feature_names, regressor.coef_, color=colors, alpha=0.8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Trọng số Hệ số của Đặc trưng (Feature Coefficients)', fontweight='bold')
axes[1].set_xlabel('Hệ số w_i')

plt.tight_layout()
plt.show()


📌 **Expected Output:**
- Đồ thị bên trái biểu diễn các điểm bám sát đường chéo $y=x$.
- Đồ thị bên phải cho thấy `R&D Spend` có hệ số dương lớn nhất tác động tới lợi nhuận.

---
### 💡 Câu hỏi Thảo luận cho Học viên:
1. Tại sao nếu không bỏ 1 cột dummy khi One-Hot Encoding thì ma trận $(\mathbf{X}^T \mathbf{X})$ sẽ không khả nghịch?
2. So sánh ưu/nhược điểm giữa Backward Elimination và Lasso Regression (L1).
